# Trend & Volume Analysis

In [0]:
%sql
-- 1: Monthly transaction volume and value trend
SELECT
  d.year, d.month,
  COUNT(*) AS txn_count,
  SUM(f.amount) AS total_value
FROM fintech_fraud_risk.gold.fact_transactions f
JOIN fintech_fraud_risk.gold.dim_date d ON f.date_key = d.date_key
GROUP BY d.year, d.month
ORDER BY d.year, d.month;


In [0]:
%sql

-- 2: 7-day moving average of daily transaction volume (smooths daily noise for the Power BI trend line)
WITH daily_counts AS (
  SELECT d.full_date, COUNT(*) AS txn_count
  FROM fintech_fraud_risk.gold.fact_transactions f
  JOIN fintech_fraud_risk.gold.dim_date d ON f.date_key = d.date_key
  GROUP BY d.full_date
)
SELECT
  full_date, txn_count,
  ROUND(AVG(txn_count) OVER (
    ORDER BY full_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
  ), 1) AS moving_avg_7d
FROM daily_counts
ORDER BY full_date;


In [0]:
%sql

-- 3: Running total of transaction value by month (cumulative revenue view)
WITH monthly AS (
  SELECT d.year, d.month, SUM(f.amount) AS monthly_value
  FROM fintech_fraud_risk.gold.fact_transactions f
  JOIN fintech_fraud_risk.gold.dim_date d ON f.date_key = d.date_key
  GROUP BY d.year, d.month
)
SELECT
  year, month, monthly_value,
  SUM(monthly_value) OVER (ORDER BY year, month) AS running_total
FROM monthly
ORDER BY year, month;